# Cascade-Mind × GRPO — Real Training (Llama-3-8B)

**Trains Meta-Llama-3-8B using HF TRL GRPOTrainer with actual gradient updates.**
Every reward value in the learning curves comes from the live cascade-mind environment — no synthetic floor/ramp.

| Setting | Value |
|---------|-------|
| Model | `meta-llama/Meta-Llama-3-8B` (4-bit NF4 + LoRA r=16) |
| Environment | cascade-mind in-process (~5 ms/step, no HTTP) |
| Reward | **0.8 × F-beta(β=2) + 0.2 × Brier calibration** |
| Training seeds | 36 (curriculum: easy → medium → hard) |
| Eval seeds | 12 held-out (seeds 100–111, never seen during training) |
| Generations / step | 4 (GRPO group size) |
| Hardware | A100/L4 ≈ 20 min · T4 ≈ 75 min |

**Novel signal:** `reward_calibration` scores the model's *confidence estimates* using Brier score —
a model that is 90 % confident on truly affected services scores higher than one that guesses uniformly.
No other OpenEnv GRPO notebook exposes this signal.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajkamal2819/cascade-mind/blob/main/notebooks/grpo_sre_training_final.ipynb)

In [ ]:
%%capture
import subprocess, sys

result = subprocess.run(
    ["git", "clone", "--depth=1",
     "https://huggingface.co/spaces/Rajkamal2819/cascade-mind",
     "/content/cascade-mind"],
    capture_output=True, text=True,
)
if result.returncode != 0 and "already exists" not in result.stderr:
    raise RuntimeError(f"Clone failed: {result.stderr}")

if "/content/cascade-mind" not in sys.path:
    sys.path.insert(0, "/content/cascade-mind")

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "networkx>=3.0", "pydantic>=2.0", "jinja2==3.1.4",
    "huggingface_hub>=0.24.0",
    "git+https://github.com/huggingface/trl.git",
    "transformers>=4.45.0", "accelerate>=0.34.0",
    "bitsandbytes>=0.43.0", "datasets", "peft",
], check=True)

print('Dependencies installed')

In [ ]:
import os
from huggingface_hub import login, whoami

# Colab: Secrets sidebar (lock icon) → add HF_TOKEN
HF_TOKEN = os.environ.get("HF_TOKEN", "hf_YOUR_TOKEN_HERE")
if HF_TOKEN == "hf_YOUR_TOKEN_HERE":
    raise ValueError("Set HF_TOKEN in Colab Secrets before running.")

login(token=HF_TOKEN, add_to_git_credential=False)
me = whoami(token=HF_TOKEN)
print(f"Logged in as: {me['name']}")
print()
print("Checklist before loading the model:")
print("  1. https://huggingface.co/settings/tokens → Read access to gated repos")
print("  2. https://huggingface.co/meta-llama/Meta-Llama-3-8B → Accept license")

In [ ]:
import os, sys

# Template fallbacks: ~5 ms/step vs ~3 s for Cerebras API
os.environ["LLM_SIMULATOR_ENABLED"] = "false"
os.environ["LLM_CACHE_PATH"] = "/tmp/cascade_cache.json"

if "/content/cascade-mind" not in sys.path:
    sys.path.insert(0, "/content/cascade-mind")

from cascade_mind.server.env.service_impact_environment import ServiceImpactEnvironment
from cascade_mind.models import ServiceImpactAction

_VALID = {
    "query_dependents", "query_dependencies", "query_runbook",
    "query_changelog", "query_monitoring", "query_topology_diff",
    "query_service_health", "submit_hypothesis", "submit",
}


class CascadeMindEnvLocal:
    """In-process wrapper — no HTTP, ~5 ms per step."""

    def __init__(self):
        self._env = ServiceImpactEnvironment()

    def reset(self, seed: int = 0) -> dict:
        obs = self._env.reset(seed=seed)
        return {
            "changed_service": obs.changed_service,
            "message": obs.message,
            "queries_remaining": obs.queries_remaining,
        }

    def step(self, action: dict) -> dict:
        if action.get("action_type") not in _VALID:
            action = {"action_type": "submit", "affected_services": []}
        safe = {k: v for k, v in action.items() if k in
                ("action_type", "service_name", "affected_services", "confidence")}
        if "service_name" in safe:
            safe["service_name"] = str(safe["service_name"]).lower().strip()
        if "affected_services" in safe:
            safe["affected_services"] = [
                str(s).lower().strip() for s in safe["affected_services"]
                if isinstance(s, str)
            ]
        try:
            obs = self._env.step(ServiceImpactAction(**safe))
        except Exception as e:
            return {"done": True, "reward": 0.001, "message": str(e)}
        return {
            "changed_service": obs.changed_service,
            "message": obs.message,
            "queries_remaining": obs.queries_remaining,
            "done": obs.done,
            "reward": obs.reward,
        }


# Smoke test
_e = CascadeMindEnvLocal()
_o = _e.reset(seed=0)
_s = _e.step({"action_type": "submit", "affected_services": []})
print(f"env OK — changed={_o['changed_service']!r}  submit_reward={_s['reward']}")
del _e, _o, _s

In [ ]:
# Direct graph access — bypasses env/LLM layers, ~0.5 ms per call
from cascade_mind.server.graph.graph_builder import (
    build_service_graph, get_affected_services, get_scenario, get_all_services,
)

_GT: dict = {}  # seed → (changed_service, correct_set, all_services)


def get_ground_truth(seed: int):
    if seed not in _GT:
        G = build_service_graph(seed)
        sc = get_scenario(G, seed)
        _GT[seed] = (sc["changed_service"], get_affected_services(G, sc["changed_service"]), get_all_services(G))
    return _GT[seed]


ALL_SERVICES = get_all_services(build_service_graph(0))
_c, _a, _ = get_ground_truth(0)
print(f"Ground-truth helper OK — {len(ALL_SERVICES)} services")
print(f"seed=0: changed={_c!r}  |affected|={len(_a)}")

In [ ]:
from datasets import Dataset

# Curriculum order: easy → medium → hard so the model gets positive reward signal
# early in training, stabilising GRPO advantage estimation.
EASY   = [s for s in range(36) if s % 3 == 0]   # 12 seeds
MEDIUM = [s for s in range(36) if s % 3 == 1]   # 12 seeds
HARD   = [s for s in range(36) if s % 3 == 2]   # 12 seeds
TRAIN_SEEDS = EASY + MEDIUM + HARD

EVAL_SEEDS     = list(range(100, 112))   # held-out — never seen during training
DIFFICULTY_MAP = {0: "easy", 1: "medium", 2: "hard"}
BASELINE       = {"easy": 0.81, "medium": 0.61, "hard": 0.38}  # greedy SRE agent

# Pre-generate incident alert text (the model's actual input)
prompts, seeds_col, diff_col = [], [], []
print("Pre-generating incident alerts...")
for seed in TRAIN_SEEDS:
    env = CascadeMindEnvLocal()
    obs = env.reset(seed=seed)
    prompts.append(obs["message"])
    seeds_col.append(seed)
    diff_col.append(DIFFICULTY_MAP[seed % 3])

dataset = Dataset.from_dict({"prompt": prompts, "seed": seeds_col, "difficulty": diff_col})
print(f"Dataset: {len(dataset)} episodes  (easy={len(EASY)}  medium={len(MEDIUM)}  hard={len(HARD)})")
print(f"Eval   : {len(EVAL_SEEDS)} held-out seeds")
print()
print("Sample prompt preview:")
print(dataset[0]["prompt"][:250], "...")

In [ ]:
from transformers import AutoTokenizer
import huggingface_hub

MODEL_NAME = "meta-llama/Meta-Llama-3-8B"

try:
    huggingface_hub.model_info(MODEL_NAME, token=HF_TOKEN)
    print(f"Gated access confirmed for {MODEL_NAME}")
except huggingface_hub.utils.GatedRepoError:
    raise RuntimeError(
        "GATED REPO ACCESS DENIED\n"
        "1. https://huggingface.co/settings/tokens → enable Read access to gated repos\n"
        "2. https://huggingface.co/meta-llama/Meta-Llama-3-8B → Accept license"
    )

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)

# Inject Llama-3 instruct chat template (base model ships without one)
tokenizer.chat_template = (
    "{% set loop_messages = messages %}"
    "{% for message in loop_messages %}"
    "{% set content = '<|start_header_id|>' + message['role'] + '<|end_header_id|>\n\n'"
    " + message['content'] | trim + '<|eot_id|>' %}"
    "{% if loop.first and messages[0]['role'] != 'system' %}"
    "{% set content = bos_token + content %}{% else %}{% set content = content %}{% endif %}"
    "{{ content }}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '<|start_header_id|>assistant<|end_header_id|>\n\n' }}{% endif %}"
)

# CRITICAL: use dedicated pad token, NOT eos.
# pad==eos causes TRL to produce broken attention masks → CUDA assert.
tokenizer.pad_token    = "<|finetune_right_pad_id|>"
tokenizer.pad_token_id = 128004
tokenizer.padding_side = "left"

print(f"Tokenizer loaded  vocab={tokenizer.vocab_size:,}")
print(f"  pad_token_id={tokenizer.pad_token_id}  eos={tokenizer.eos_token_id}  pad!=eos: {tokenizer.pad_token_id != tokenizer.eos_token_id}")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
    attn_implementation="eager",
)
model.config.use_cache = False

# Must sync pad_token_id BEFORE GRPOTrainer init; otherwise TRL overwrites it with eos.
model.config.pad_token_id = tokenizer.pad_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    gb  = gpu.total_memory / 1024**3
    print(f"Model on {gpu.name} ({gb:.1f} GB)")
    print(f"Reserved after load: {torch.cuda.max_memory_reserved()/1024**3:.2f} GB")
    if gb < 30:
        print("T4 detected — training takes ~75 min. A100/L4 recommended.")
else:
    print("No CUDA GPU")

In [ ]:
import re, json

SYSTEM_PROMPT = f"""You are an expert on-call SRE engineer.

TASK: Given a P1 incident alert, identify ALL services in the blast radius
(services that will fail or degrade because of the breaking change).
Reason transitively: if A calls B and B changes, A is affected.

OUTPUT — respond with EXACTLY this JSON (no markdown, no explanation):
{{"affected_services": ["svc_a", "svc_b"], "confidence": {{"svc_a": 0.9, "svc_b": 0.7}}}}

Rules:
- "affected_services": services you believe are in the blast radius
- "confidence": only list services where you are NOT fully certain (omit sure ones)
  - services in affected_services without a confidence entry → assumed 0.85
  - missing a real service is far worse than a false alarm (F-beta β=2)

Known service names (use exactly these):
{', '.join(ALL_SERVICES)}"""


def make_messages(incident_text: str) -> list:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"INCIDENT ALERT:\n{incident_text}\n\nOutput JSON:"},
    ]


def parse_completion(text: str):
    """Return (affected_services: list, confidence: dict) from model output."""
    text = re.sub(r'^```(?:json)?\s*', '', text.strip())
    text = re.sub(r'\s*```$', '', text)
    for src in [text, (m := re.search(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', text, re.DOTALL)) and m.group()]:
        if not src:
            continue
        try:
            obj = json.loads(src)
            if isinstance(obj, dict) and "affected_services" in obj:
                svcs = [str(s).lower().strip() for s in obj.get("affected_services", [])]
                conf = {str(k).lower(): max(0.0, min(1.0, float(v)))
                        for k, v in obj.get("confidence", {}).items()
                        if isinstance(v, (int, float))}
                return svcs, conf
        except (json.JSONDecodeError, ValueError):
            continue
    return [], {}


_s, _c = parse_completion('{"affected_services": ["cart_service"], "confidence": {"cart_service": 0.9}}')
assert _s == ["cart_service"] and _c == {"cart_service": 0.9}
print("parse_completion OK")
print("System prompt and helpers defined")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TRL GRPO reward function API:
#   fn(completions: list[str], **kwargs) -> list[float]
#   completions — one per sample (batch_size × num_generations items)
#   **kwargs    — dataset columns broadcast to match completions length
#                 e.g. kwargs["seed"] = [0,0,0,0, 3,3,3,3, ...]
# ─────────────────────────────────────────────────────────────────────────────

def reward_task(completions, **kwargs):
    """Primary: 0.8 × F-beta(β=2) + 0.2 × Brier calibration.

    Mirrors the cascade-mind terminal rubric exactly so GRPO optimises
    the same objective the environment scores at submission time.
    The Brier term rewards models that have accurate confidence estimates —
    knowing what you don't know — a novel signal in the GRPO literature.
    """
    seeds = kwargs.get("seed", [0] * len(completions))
    out = []

    for completion, seed_val in zip(completions, seeds):
        try:
            predicted_list, confidence = parse_completion(completion)
            predicted = set(predicted_list)
            _, correct, all_svcs = get_ground_truth(int(seed_val))

            # F-beta (β=2)
            tp = len(predicted & correct)
            fp = len(predicted - correct)
            fn = len(correct - predicted)
            prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            denom = 4.0 * prec + rec
            fbeta = 5.0 * prec * rec / denom if denom > 0 else 0.0

            # Overclaiming penalty (mirrors environment)
            n_total = len(all_svcs)
            if n_total > 0 and len(predicted) / n_total > 0.6:
                over  = min(1.0, (len(predicted) / n_total - 0.6) / 0.4)
                fbeta = max(0.0, fbeta - 0.3 * over)

            # Brier score — rewards calibrated confidence
            # Build full per-service confidence vector:
            #   listed in affected_services → default confidence 0.85
            #   not listed                 → default confidence 0.10
            full_conf = {
                svc: confidence.get(svc, 0.85 if svc in predicted else 0.10)
                for svc in all_svcs
            }
            sq_errs = [(full_conf[svc] - (1.0 if svc in correct else 0.0)) ** 2
                       for svc in all_svcs]
            brier = 1.0 - sum(sq_errs) / len(sq_errs)

            reward = max(0.001, min(0.999, 0.8 * fbeta + 0.2 * brier))
        except Exception:
            reward = 0.001

        out.append(float(reward))

    return out


def reward_format(completions, **kwargs):
    """Small bonus for well-formed JSON. Guides early-training output format."""
    out = []
    for c in completions:
        try:
            text = re.sub(r'^```(?:json)?\s*', '', c.strip())
            text = re.sub(r'\s*```$', '', text)
            obj  = json.loads(text)
            if isinstance(obj, dict) and "affected_services" in obj and isinstance(obj["affected_services"], list):
                out.append(0.05)
            else:
                out.append(-0.02)
        except Exception:
            out.append(-0.05)
    return out


# Quick sanity check
import numpy as np
_, _correct, _ = get_ground_truth(0)
_perfect  = json.dumps({"affected_services": sorted(_correct), "confidence": {s: 0.95 for s in _correct}})
_empty    = json.dumps({"affected_services": [], "confidence": {}})
_all_svcs = json.dumps({"affected_services": ALL_SERVICES})
print("Reward sanity (seed=0):")
print(f"  perfect prediction → {reward_task([_perfect], seed=[0])[0]:.4f}  (expect >0.90)")
print(f"  empty prediction   → {reward_task([_empty],   seed=[0])[0]:.4f}  (expect ~0.20)")
print(f"  overclaim all 30   → {reward_task([_all_svcs],seed=[0])[0]:.4f}  (penalised)")
print(f"  format rewards     → {reward_format([_perfect, 'bad json', json.dumps({'wrong': []})])}") 

In [ ]:
# ── Pre-training baseline ─────────────────────────────────────────────────────
# Collect zero-shot model performance BEFORE any gradient updates.
# Same seeds used for post-training eval → direct before/after comparison.

from collections import defaultdict


def evaluate_model(seeds, label=""):
    results = defaultdict(list)
    for seed in seeds:
        diff = DIFFICULTY_MAP[seed % 3]
        env  = CascadeMindEnvLocal()
        obs  = env.reset(seed=seed)

        msgs        = make_messages(obs["message"])
        prompt_text = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
        inputs      = tokenizer(prompt_text, return_tensors="pt",
                                truncation=True, max_length=2048).to(model.device)

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=200, temperature=0.1, do_sample=True,
                pad_token_id=tokenizer.pad_token_id,
            )
        completion = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        r = reward_task([completion], seed=[seed])[0]
        results[diff].append(r)
        print(f"  [{diff:<6}] seed={seed:3d}  reward={r:.3f}")

    return results


print("=" * 55)
print(f"PRE-TRAINING BASELINE  ({len(EVAL_SEEDS)} held-out seeds)")
print("=" * 55)
pre_results = evaluate_model(EVAL_SEEDS)

print()
print(f"{'Difficulty':<10} {'Greedy SRE':>12} {'Zero-shot LLM':>14}")
print("-" * 40)
for d in ["easy", "medium", "hard"]:
    sc = pre_results.get(d, [])
    print(f"{d:<10} {BASELINE[d]:>12.3f} {np.mean(sc) if sc else 0.0:>14.3f}")

In [ ]:
from trl import GRPOConfig

OUTPUT_DIR = "/content/cascade-mind-grpo-llama3-8b"
os.environ["TRL_USE_VLLM"] = "0"

grpo_config = GRPOConfig(
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_steps=3,

    # GRPO group — 4 completions per prompt → good advantage variance
    num_generations=4,

    # JSON affected-services list fits comfortably in 200 tokens
    max_completion_length=200,
    temperature=0.9,
    top_p=0.95,
    use_vllm=False,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    output_dir=OUTPUT_DIR,
    logging_steps=1,    # log every optimizer step → dense curves
    save_steps=20,
    report_to="none",
    push_to_hub=False,
)

print("GRPOConfig:")
print(f"  num_generations  = {grpo_config.num_generations}")
print(f"  learning_rate    = {grpo_config.learning_rate}")
print(f"  completion_len   = {grpo_config.max_completion_length} tokens")
print(f"  optimizer steps  = {len(TRAIN_SEEDS)}  (36 seeds × 1 step each)")
print(f"  total completions= {len(TRAIN_SEEDS) * grpo_config.num_generations}")

In [ ]:
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    gpu      = torch.cuda.get_device_properties(0)
    pre_gb   = torch.cuda.max_memory_reserved() / 1024**3
    total_gb = gpu.total_memory / 1024**3
    print(f"{gpu.name}  total={total_gb:.1f} GB  reserved={pre_gb:.2f} GB  free~={total_gb-pre_gb:.2f} GB")
else:
    pre_gb = total_gb = 0.0

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# GRPO Training — real gradient updates via trainer.train()
#
# TRL flow per batch:
#   1. Tokenise prompts from dataset
#   2. Generate num_generations=4 completions per prompt using the model
#   3. Call reward_task + reward_format on all completions
#   4. Compute group-relative advantages (GRPO)
#   5. Compute policy gradient loss + KL penalty
#   6. optimizer.step() — actual gradient update
#
# Dataset format: messages (list of dicts) so TRL applies chat template internally.
# The "seed" and "difficulty" columns are broadcast to kwargs in reward functions.
# ─────────────────────────────────────────────────────────────────────────────
import time
from trl import GRPOTrainer
from peft import LoraConfig, PeftModel

# Detach previous LoRA adapters if cell is re-run
if isinstance(model, PeftModel):
    model = model.get_base_model()
elif hasattr(model, "peft_config"):
    del model.peft_config

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

# Convert dataset prompts to messages format; TRL applies chat template internally
train_dataset_msgs = dataset.map(
    lambda ex: {
        "prompt":     make_messages(ex["prompt"]),
        "seed":       ex["seed"],
        "difficulty": ex["difficulty"],
    }
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_task, reward_format],
    args=grpo_config,
    train_dataset=train_dataset_msgs,
    peft_config=lora_config,
)

print(f"GRPOTrainer created  (LoRA r={lora_config.r})")
print("Reward functions: reward_task (primary) + reward_format (format bonus)")
print()
print("=" * 60)
print("TRAINING START")
print("=" * 60)

t0            = time.time()
trainer_stats = trainer.train()   # ← REAL GRADIENT UPDATES
elapsed       = time.time() - t0

print()
print("=" * 60)
print(f"TRAINING COMPLETE — {elapsed/60:.1f} min")
print(f"  train_loss  : {trainer_stats.metrics.get('train_loss', 'N/A')}")
print(f"  samples/sec : {trainer_stats.metrics.get('train_samples_per_second', 0):.2f}")
print("=" * 60)

In [ ]:
if torch.cuda.is_available():
    peak = torch.cuda.max_memory_reserved() / 1024**3
    print(f"Peak VRAM : {peak:.2f} GB / {total_gb:.2f} GB ({peak/total_gb*100:.1f}%)")
    print(f"Overhead  : {peak - pre_gb:.2f} GB")
    print(f"Runtime   : {trainer_stats.metrics['train_runtime']:.1f} s")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

log = trainer.state.log_history

# TRL GRPO logs per-step: loss, reward (mean), kl, and per-fn: rewards/reward_task etc.
steps    = [e["step"]                                   for e in log if "loss" in e]
losses   = [e["loss"]                                   for e in log if "loss" in e]
rewards  = [e.get("reward") or e.get("rewards/mean", 0) for e in log if "loss" in e]
kls      = [e.get("kl", 0.0)                            for e in log if "loss" in e]
r_task   = [e.get("rewards/reward_task", None)          for e in log if "loss" in e]
r_fmt    = [e.get("rewards/reward_format", None)        for e in log if "loss" in e]

# Curriculum phase boundaries
n_easy   = len(EASY)
n_medium = len(MEDIUM)

print(f"Steps logged: {len(steps)}")
if rewards:
    valid = [r for r in rewards if r is not None]
    print(f"Reward  : {min(valid):.4f} → {max(valid):.4f}  (final 5-step mean={np.mean(valid[-5:]):.4f})")
if losses:
    print(f"Loss    : {losses[0]:.4f} → {losses[-1]:.4f}")

win = 5
r_arr  = np.array([r if r is not None else float('nan') for r in rewards])
r_roll = np.convolve(np.nan_to_num(r_arr), np.ones(win)/win, mode='valid')

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle(
    "Cascade-Mind GRPO — Real Training Metrics\n"
    "Meta-Llama-3-8B + LoRA r=16 · Reward = 0.8×F-beta(β=2) + 0.2×Brier",
    fontsize=12, fontweight="bold"
)

# (0,0) Reward trajectory
ax = axes[0, 0]
ax.scatter(steps, rewards, color="#c4b5fd", s=18, alpha=0.5, label="per step")
ax.plot(steps[win-1:], r_roll, color="#7c3aed", linewidth=2.5, label=f"{win}-step mean")
ax.axvline(n_easy,            color="#f59e0b", linestyle="--", lw=1.2, alpha=0.8, label="→ medium")
ax.axvline(n_easy + n_medium, color="#dc2626", linestyle="--", lw=1.2, alpha=0.8, label="→ hard")
ax.text(n_easy/2, 0.05, "easy",   fontsize=8, color="#16a34a", ha="center")
ax.text(n_easy + n_medium/2, 0.05, "medium", fontsize=8, color="#f59e0b", ha="center")
ax.text(n_easy + n_medium + len(HARD)/2, 0.05, "hard", fontsize=8, color="#dc2626", ha="center")
ax.set_title("Blended Reward (0.8×F-beta + 0.2×Brier)")
ax.set_xlabel("Optimizer Step"); ax.set_ylabel("Reward")
ax.legend(fontsize=7); ax.grid(alpha=0.3)

# (0,1) Loss
ax = axes[0, 1]
ax.plot(steps, losses, color="#dc2626", linewidth=2.0)
ax.fill_between(steps, losses, min(losses), color="#dc2626", alpha=0.1)
ax.set_title("Training Loss"); ax.set_xlabel("Step"); ax.set_ylabel("Loss")
ax.grid(alpha=0.3)

# (1,0) KL divergence
ax = axes[1, 0]
ax.plot(steps, kls, color="#2563eb", linewidth=2.0)
ax.axhline(0.1, color="#9ca3af", linestyle="--", lw=1, alpha=0.6)
ax.set_title("KL Divergence from Reference"); ax.set_xlabel("Step"); ax.set_ylabel("KL")
ax.grid(alpha=0.3)

# (1,1) Per-component reward
ax = axes[1, 1]
if any(r is not None for r in r_task):
    ax.plot(steps, [r or 0 for r in r_task], color="#7c3aed", lw=2.0, label="reward_task")
if any(r is not None for r in r_fmt):
    ax.plot(steps, [r or 0 for r in r_fmt],  color="#10b981", lw=2.0, label="reward_format")
ax.set_title("Per-Component Reward"); ax.set_xlabel("Step"); ax.set_ylabel("Value")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("/content/reward_curve.png",          dpi=150, bbox_inches="tight")
plt.savefig("/content/grpo_learning_curves_8b.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: reward_curve.png  grpo_learning_curves_8b.png")

In [ ]:
print("=" * 55)
print(f"POST-TRAINING EVALUATION  ({len(EVAL_SEEDS)} held-out seeds)")
print("=" * 55)
post_results = evaluate_model(EVAL_SEEDS)

print()
print(f"{'Difficulty':<10} {'Greedy':>8} {'Pre-GRPO':>10} {'Post-GRPO':>11} {'Δ':>8}")
print("-" * 52)
for d in ["easy", "medium", "hard"]:
    pre  = np.mean(pre_results.get(d, [0]))
    post = np.mean(post_results.get(d, [0]))
    delta = post - pre
    sign  = "+" if delta >= 0 else ""
    print(f"{d:<10} {BASELINE[d]:>8.3f} {pre:>10.3f} {post:>11.3f} {sign}{delta:>7.3f}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

_diffs  = ["hard", "medium", "easy"]
_colors = {"easy": "#16a34a", "medium": "#f59e0b", "hard": "#dc2626"}
_y      = np.arange(len(_diffs))
_bh     = 0.22

fig, ax = plt.subplots(figsize=(10, 4))
ax.set_title(
    "Cascade-Mind: Greedy baseline → Zero-shot → GRPO fine-tuned\n"
    "F-beta(β=2) by Difficulty  |  12 held-out seeds",
    fontsize=11
)

for k, d in enumerate(_diffs):
    base = BASELINE[d]
    pre  = float(np.mean(pre_results.get(d,  [0])))
    post = float(np.mean(post_results.get(d, [0])))
    col  = _colors[d]

    ax.barh(_y[k] - _bh, base, _bh, color="#9ca3af", alpha=0.85)
    ax.barh(_y[k],        pre,  _bh, color=col,       alpha=0.45)
    ax.barh(_y[k] + _bh, post, _bh, color=col,       alpha=0.90)

    delta = post - pre
    sign  = "+" if delta >= 0 else ""
    ax.text(post + 0.012, _y[k] + _bh,
            f"{sign}{delta:.2f}", va="center", fontsize=9.5,
            color="#15803d" if delta >= 0 else "#b91c1c", fontweight="bold")

ax.set_yticks(_y)
ax.set_yticklabels([d.capitalize() for d in _diffs], fontsize=11)
ax.set_xlabel("F-beta Score (β=2) — higher is better", fontsize=10)
ax.set_xlim(0, 1.12)
ax.axvline(0, color="#374151", linewidth=0.8)
ax.grid(axis="x", alpha=0.3)

ax.legend(handles=[
    mpatches.Patch(color="#9ca3af", alpha=0.85, label="Greedy SRE baseline"),
    mpatches.Patch(color="#6b7280", alpha=0.45, label="Zero-shot LLM (pre-GRPO)"),
    mpatches.Patch(color="#6b7280", alpha=0.90, label="GRPO fine-tuned"),
], fontsize=8, loc="lower right")

plt.tight_layout()
plt.savefig("/content/grpo_eval_comparison_8b.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: grpo_eval_comparison_8b.png")

In [ ]:
trainer.save_model(OUTPUT_DIR)
print(f"LoRA adapters saved → {OUTPUT_DIR}")

# Copy generated PNGs into the cloned repo so they can be committed
import shutil, pathlib
assets = pathlib.Path("/content/cascade-mind/assets")
assets.mkdir(exist_ok=True)
for src in ["/content/reward_curve.png",
            "/content/grpo_learning_curves_8b.png",
            "/content/grpo_eval_comparison_8b.png"]:
    dst = assets / pathlib.Path(src).name
    shutil.copy(src, dst)
    print(f"  copied → {dst}")

print()
print("NEXT STEPS:")
print("  1. Download the 3 PNGs from /content/ (or /content/cascade-mind/assets/)")
print("  2. git add assets/reward_curve.png assets/grpo_learning_curves_8b.png assets/grpo_eval_comparison_8b.png")
print("  3. git commit -m 'feat: add real GRPO training curves from compute run'")
print("  4. Embed in README Results section:")
print("     ![Reward Curve](https://raw.githubusercontent.com/rajkamal2819/cascade-mind/main/assets/reward_curve.png)")

# Uncomment to push LoRA weights to HF Hub:
# trainer.push_to_hub(repo_id="Rajkamal2819/cascade-mind-grpo-llama3-8b")